<a href="https://colab.research.google.com/github/guirco/ufpel-pdi-2026-1/blob/main/2026_1_LAB5_Remocao_Ruido_Melhoria_Qualidade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB5 — Pipeline de Processamento para Remoção de Ruído e Melhoria de Qualidade Visual

Disciplina: **Processamento Digital de Imagens (PDI)** – UFPel  
Professor: **Guilherme Corrêa**  

Vamos exercitar os conceitos estudados até o momento criando um pipeline de processamento de imagens.

Um pipeline de PDI é um fluxo com diversas operações executadas sequencialmente. No caso deste laboratório, vamos executar diversas operações para tentar melhorar a qualidade de uma imagem que tem baixa resolução, sofreu efeito de ruído periódico e também sofreu efeito de ruído sal & pimenta.

Use a imagem `tuyuka_lq.png` está disponível no repositório para os exercícios.

---

## Objetivos  

1. Aplicar conceitos estudados até este momento para melhorar a qualidade de uma imagem que tem baixa resolução, sofreu efeito de ruído periódico e também sofreu efeito de ruído sal & pimenta.
2. Você deve escolher quais passos de PDI aplicar e em qual ordem, considerando tudo o que estudamos até aqui (interpolação, realce, suavização, filtragem no domínio espacial, filtragem no domínio de transformadas...
3. **IMPORTANTE: a ordem de aplicação das técnicas vai influenciar muito no resultado final! Escolha sabiamente!**

---

## Bibliotecas úteis
Se estiver no Colab, rode a célula de instalação uma única vez.

In [ ]:
# Se necessário no Colab, descomente a linha abaixo:
#!pip -q install numpy matplotlib scikit-image imageio

In [ ]:
# (execute uma vez)
!pip -q install ipywidgets==8.1.2 scikit-image==0.24.0 opencv-python-headless==4.10.0.84
from google.colab import output
output.enable_custom_widget_manager()

import numpy as np
import matplotlib.pyplot as plt
import cv2
from ipywidgets import interact, IntSlider
from skimage import filters, img_as_float
from skimage import io, color, img_as_ubyte
from google.colab import files



## Upload de uma imagem
Usando `files` do `google.colab` para fazer upload de uma imagem.

In [ ]:

print("Faça upload de uma imagem (JPG/PNG).")
up = files.upload()
if not up:
    raise RuntimeError("Nenhum arquivo enviado.")

# Nome do arquivo
fname = next(iter(up))

# Ler imagem colorida (BGR)
img_bgr = cv2.imdecode(np.frombuffer(up[fname], np.uint8), cv2.IMREAD_COLOR)
if img_bgr is None:
    raise RuntimeError("Falha ao ler a imagem.")

# Converter para escala de cinza (uint8, faixa 0–255)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_orig = img_gray.astype(np.uint8)

# Mostrar
plt.imshow(img_orig, cmap='gray', vmin=0, vmax=255)
plt.axis('off')
plt.title('Imagem (grayscale, 0–255)')
plt.show()

print("Dimensão:", img_orig.shape, " | dtype:", img_orig.dtype, " | Faixa:", (img_orig.min(), img_orig.max()))


# 🖼️ Tarefa A — Transformar a imagem para o domínio das frequências (DFT) e analisar visualmente o espectro de Fourier

1) Transformada de Fourier bidimensional e visualização do espectro

Você deve:

- Utilizar a imagem em escala de cinza aberta no bloco anterior.
- Calcular a transformada de Fourier bidimensional (DFT) da imagem, utilizando as funções prontas do módulo numpy.fft.
- Centralizar o espectro de frequências com `np.fft.fftshift`, de modo que as baixas frequências fiquem no centro.
- Alterar o espectro de frequências para escala logarítmica `(np.log1p(np.abs(...)))` para melhor visualização.
- Mostrar a imagem que representa o espectro de frequências e a imagem que representa o espectro de frequências alterado para a escala logarítmica.

**Passo adicional para implementar:**

- Normalize o espectro em escala logarítmica para o intervalo de 0 a 255 e mostre essa versão normalizada como uma imagem do tipo uint8.

💡 Dicas:

* `np.fft.fft2(img)` transforma a imagem do domínio espacial para o espaço das
frequências com a DFT bidimensional.
* `np.fft.fftshift()` reorganiza os quadrantes do espectro, movendo a frequência zero para o centro da imagem.
* `np.log1p(np.abs(F))` aplica log para melhor visualização da magnitude do espectro de frequências.
* `cv2.normalize()` ou uma normalização manual podem ser usados para converter o espectro logarítmico para o intervalo [0, 255].

# 🖼️ Tarefa B — Transformar a imagem de volta para o domínio espacial (IDFT)

Você deve:

- Utilizar a representação no domínio das frequências do bloco anterior.
- Calcular a transformada de Fourier inversa bidimensional (IDFT) da imagem, utilizando as funções prontas do módulo numpy.fft.
- Descentralizar o espectro de frequências com `np.fft.ifftshift()`, de modo que as baixas frequências voltem ao canto esquerdo superior.
- Mostrar a imagem que representa o espectro de frequências e a imagem que representa o espectro de frequências alterado para a escala logarítmica.

**Passo adicional para implementar:**

- Mostre também a imagem reconstruída no domínio espacial e calcule a diferença absoluta entre a imagem original e a imagem reconstruída.

💡 Dicas:

* `np.fft.ifft2(img)` transforma a imagem do domínio das frequências de volta para o domínio espacial.
* `np.fft.ifftshift()` reorganiza os quadrantes do espectro, movendo a frequência zero de volta para o canto esquerdo superior.
* `np.abs()` pode ser usado para obter a parte visualizável da imagem reconstruída.
* `cv2.absdiff()` ou `np.abs(img_orig - img_reconstruida)` podem ser usados para calcular a diferença entre as imagens.

---
# 🖼️ Tarefa C — Suavização com Filtro Rejeita-Banda

Você deve:

- Usar a imagem em escala de cinza e o espectro de Fourier F já calculado com **fft2c**.
- Criar uma máscara rejeita-banda circular com a função **band_reject_mask(shape, D0, pos)**.
- Definir D0 como o raio da região de frequências que será removida.
- Definir pos para indicar a posição da frequência a ser rejeitada no espectro.
- Criar duas máscaras: uma em +pos e outra em -pos, pois o ruído periódico costuma aparecer em pares simétricos.
- Combinar as duas máscaras em uma única máscara H.
- Aplicar o filtro no espectro usando **G = F * filtro_H**.
- Exibir lado a lado:
  - o espectro original em escala logarítmica;
  - a máscara rejeita-banda;
  - o espectro filtrado.

**Passo adicional:**

- Após aplicar o filtro rejeita-banda, teste pelo menos dois valores diferentes para D0 e pos, por exemplo D0 = 3, pos = 18 e D0 = 6, pos = 18. Compare visualmente as máscaras e os espectros filtrados para observar como o tamanho e a posição da região rejeitada afetam a remoção das frequências.

💡 Dicas:

- `np.log1p(np.abs(F))` pode ser usado para visualizar melhor o espectro de Fourier.
- A variável D0 controla o tamanho da região rejeitada.
- A variável pos controla a posição da frequência que será removida.
- A máscara H deve conter valor 0 nas frequências rejeitadas e valor 1 nas frequências mantidas.
- Use `plt.subplot()` para comparar os resultados lado a lado.

---
# 🖼️ Tarefa D — Veja o Resultado da Filtragem Rejeita-Banda

Você deve:

- Utilizar o espectro filtrado G, obtido após aplicar o filtro rejeita-banda.
- Visualizar esse espectro em escala logarítmica com `np.log1p(np.abs(G))`.
- Aplicar a transformada inversa com `ifft2c(G)` para retornar a imagem ao domínio espacial.
- Converter o resultado para uma imagem válida, limitando os valores entre 0 e 255 com `np.clip()`.
- Exibir lado a lado:
  - a imagem original;
  - o espectro filtrado G em escala logarítmica;
  - a imagem final após a filtragem rejeita-banda.

**Passo adicional para implementar:**

- Calcule a diferença absoluta entre a imagem original e a imagem após o filtro rejeita-banda. Mostre essa diferença como uma quarta imagem para visualizar quais regiões foram mais alteradas pela filtragem.

💡 Dicas:

- G representa o espectro da imagem depois da aplicação do filtro rejeita-banda.
- `np.log1p(np.abs(G))` melhora a visualização do espectro, pois os valores de magnitude podem ser muito altos.
- `ifft2c(G)` retorna a imagem filtrada para o domínio espacial.
- `np.abs()` é usado para obter uma imagem visualizável a partir do resultado da transformada inversa.
- `np.clip(img, 0, 255).astype(np.uint8)` garante que a imagem fique no intervalo correto de intensidades.
- Use `plt.subplot()` para comparar os resultados lado a lado.

---
# 🖼️ Tarefa E — Filtro de Mediana para Remover Ruído Sal & Pimenta

Você deve:

- Utilizar a imagem obtida após a filtragem rejeita-banda.
- Aplicar um filtro de mediana 3x3 usando `cv2.medianBlur()`.
- Armazenar o resultado na variável img_mediana.
- Exibir lado a lado:
  - a imagem original;
  - a imagem após o filtro rejeita-banda;
  - a imagem após o filtro de mediana.

**Passo adicional:**

- Aplique também um filtro de mediana 5x5. Mostre os dois resultados lado a lado e compare qual deles remove melhor o ruído, observando se houve perda de detalhes na imagem.

💡 Dicas:

- `cv2.medianBlur(img, ksize=3)` aplica o filtro de mediana usando uma janela 3x3.
- O valor de ksize precisa ser ímpar, como 3, 5 ou 7.
- O filtro de mediana substitui cada pixel pelo valor mediano de sua vizinhança.
- Use `plt.subplot()` para comparar visualmente os resultados.

---
# 🖼️ Tarefa F — Filtro de Realce (Laplaciano) para Melhorar Contornos/Bordas

Você deve:

- Utilizar a imagem resultante do filtro de mediana.
- Implementar uma função chamada **realce_laplaciano_3x3(figura, k)**.
- Converter a imagem para np.float32 para permitir os cálculos.
- Aplicar padding refletido nas bordas da imagem.
- Calcular o Laplaciano usando uma máscara 3x3 com centro -8 e vizinhos com peso 1.
- Aplicar o realce usando **figura - k * lap**.
- Gerar a imagem **img_laplaciano** com **k = 0.2**.
- Exibir lado a lado:
  - a imagem original;
  - a imagem após o filtro rejeita-banda;
  - a imagem após o filtro de mediana;
  - a imagem após o realce Laplaciano.

**Passo adicional:**
- Teste outros valores de k, como 0.1 e 0.5. Mostre os resultados lado a lado para comparar como o valor de k altera a intensidade do realce das bordas.

💡 Dicas:

- O filtro Laplaciano realça regiões onde há mudanças bruscas de intensidade.
- A máscara 3x3 usada no código considera os 8 vizinhos ao redor do pixel central.
- O parâmetro k controla a intensidade do realce.
- Valores pequenos de k produzem um realce mais suave.
- Valores altos de k podem destacar mais as bordas, mas também podem aumentar ruídos.
- Use `np.clip(img, 0, 255).astype(np.uint8)` caso a imagem final fique com valores fora do intervalo correto.
- Use `plt.subplot()` para comparar todas as etapas do processamento.

---
# 🖼️ Tarefa G — Comparação entre filtro passa-baixa e filtro passa-alta no domínio da frequência

Você deve:

- Usar a imagem em escala de cinza já carregada.
- Calcular a Transformada de Fourier 2D e centralizar o espectro.
- Criar duas máscaras circulares:
  - passa-baixa, mantendo as frequências próximas ao centro;
  - passa-alta, mantendo as frequências mais distantes do centro.
- Aplicar cada máscara no espectro da imagem.
- Retornar os dois resultados para o domínio espacial com a transformada inversa.
- Exibir lado a lado:
  - imagem original;
  - imagem com filtro passa-baixa;
  - imagem com filtro passa-alta.

**Passo adicional:**

- Teste dois valores diferentes de raio para as máscaras, por exemplo raio = 20 e raio = 60, e compare como o tamanho do raio altera o resultado final.

💡 Dicas:

Use np.ogrid para criar uma máscara circular.
A distância ao centro pode ser calculada por:

**`distancia = np.sqrt((X - centro_x)**2 + (Y - centro_y)**2)`**

Para o filtro passa-baixa:

**mascara_pb = distancia <= raio**

Para o filtro passa-alta:

**mascara_pa = distancia > raio**

Depois de aplicar a máscara, use a transformada inversa para retornar a imagem ao domínio espacial.